In [ ]:
# Завдання 2

import requests
from datetime import datetime, timezone
import pandas as pd

API_URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange"
NAMES = {"exchangedate": "exchange_date", "cc": "currency"}

def fetch_rates(rate_date, currencies=("USD", "EUR")):
    response = requests.get(
        API_URL,
        params={"date": rate_date.strftime("%Y%m%d"), "json": ""},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    if not data:
        raise ValueError(f"API не повернув курсів за {rate_date}")
    raw = pd.DataFrame(data)
    rates = raw[raw["cc"].isin(currencies)].rename(columns=NAMES)
    rates["exchange_date"] = pd.to_datetime(rates["exchange_date"], format="%d.%m.%Y").dt.date
    rates = rates.sort_values(["currency"]).reset_index(drop=True)
    return rates[["exchange_date", "currency", "rate"]]


today_rates = fetch_rates(datetime.now(timezone.utc))
print(today_rates)


  exchange_date currency     rate
0    2026-11-27      EUR  49.4942
1    2026-11-27      USD  45.1064


In [36]:
# Завдання 3


list_october_rates = []
for r in pd.date_range(start="2026-10-01", end="2026-10-31"):
    list_october_rates.append(fetch_rates(r))

october_rates = pd.concat(list_october_rates, ignore_index=True)
october_rates = october_rates.sort_values(["exchange_date", "currency"]).reset_index(drop=True)